<a href="https://colab.research.google.com/github/KIMSUNGROK/StockAnalysisInPython/blob/master/Antigravity_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(f"Is GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
        print(f"GPU Name: {torch.cuda.get_device_name(0)}")


Is GPU available: True
GPU Name: Tesla T4


In [ ]:
# Step 1: Install dependencies
!pip install -q jupyter_http_over_ws
!jupyter serverextension enable --py jupyter_http_over_ws


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.5 MB/s eta 0:00:00
Enabling: jupyter_http_over_ws
- Writing config: /root/.jupyter
    - Validating...
      jupyter_http_over_ws 0.0.7 OK


In [ ]:


!jupyter notebook --NotebookApp.allow_origin='https://colab.research.google.com' --port=8888 --NotebookApp.port_retries=0 --no-browser --ip=0.0.0.0 --NotebookApp.token='colab_gpu_2026' &

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
[I 18:32:40.649 NotebookApp] Writing notebook server cookie secret to /root/.local/share/jupyter/runtime/notebook_cookie_secret

  _   _          _      _
 | | | |_ __  __| |__ _| |_ ___
 | |_| | '_ \/ _` / _` |  _/ -_)
  \___/| .__/\__,_\__,_|\__\___|
       |_|
                       
Read the migration plan to Notebook 7 to learn about the new features and the actions to take if you are using extensions.

https://jupyter-notebook.readthedocs.io/en/latest/migrate_to_notebook7.html

Please note that updating to Notebook 7 might break some of your extensions.

[I 18:32:41.007 NotebookApp] Loading IPython parallel extension
jupyter_http_over_ws extension initialized. Listening on /http_over_websocket
[I 18:32:41.095 NotebookApp] [Jupytext Server Extension] Deriving a TextFileContentsManager from LargeFileManager
[C 18:32:42.055 NotebookApp] R

In [ ]:
# Colab GPU Remote Access Setup via cloudflared
import subprocess
import os
import time

# Install cloudflared
print('Installing cloudflared...')
subprocess.run(['apt-get', 'install', '-y', 'wget'], capture_output=True)
subprocess.run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], capture_output=True)
subprocess.run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], capture_output=True)
print('cloudflared installed!')

# Kill any existing jupyter servers on port 9999
subprocess.run(['bash', '-c', 'kill $(lsof -t -i:9999) 2>/dev/null'], capture_output=True)

# Start Jupyter server in background
print('Starting Jupyter server on port 9999...')
proc = subprocess.Popen(
        ['jupyter', 'notebook',
              '--NotebookApp.allow_origin=*',
              '--NotebookApp.token=colab_gpu_2026',
              '--port=9999',
              '--no-browser',
              '--ip=0.0.0.0',
              '--NotebookApp.disable_check_xsrf=True'],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE
        )
time.sleep(3)
print('Jupyter server started!')

# Start cloudflared tunnel
print('Starting cloudflared tunnel...')
print('=' * 60)
print('Wait for the tunnel URL to appear below...')
print('=' * 60)
os.system('cloudflared tunnel --url http://localhost:9999 2>&1 | grep -m1 "trycloudflare.com"')

Installing cloudflared...
cloudflared installed!
Starting Jupyter server on port 9999...
Jupyter server started!
Starting cloudflared tunnel...
Wait for the tunnel URL to appear below...


0

In [ ]:
!cloudflared --version

cloudflared version 2026.3.0 (built 2026-03-09-14:08 UTC)


In [ ]:
!cloudflared tunnel --url http://localhost:9999 2>&1 | grep "trycloudflare.com"

2026-03-12T18:38:17Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-12T18:38:22Z INF |  https://benz-atlantic-fed-earnings.trycloudflare.com                                      |
2026-03-12T18:41:05Z ERR Request failed error="Unable to reach the origin service. The service may be down or it may not be responding to traffic from cloudflared: dial tcp [::1]:9999: connect: connection refused" connIndex=0 dest=https://benz-atlantic-fed-earnings.trycloudflare.com/api/kernels?token=colab_gpu_2026 event=0 ip=198.41.200.23 type=http
2026-03-12T18:41:05Z ERR Request failed error="Unable to reach the origin service. The service may be down or it may not be responding to traffic from cloudflared: dial tcp [::1]:9999: connect: connection refused" connIndex=0 dest=https://benz-atlantic-fed-earnings.trycloudflare.com/api/kernels?token=colab_gpu_2026 event=0 ip=198.41.200.23 type=http


In [ ]:
!grep trycloudflare /tmp/cloudflared.log

2026-03-12T18:45:58Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-12T18:46:03Z INF |  https://practice-operating-flooring-gathering.trycloudflare.com                           |


In [ ]:
!pkill -f cloudflared; pkill -f jupyter-notebook; sleep 2; nohup jupyter notebook --port=9999 --no-browser --ip=0.0.0.0 --NotebookApp.allow_origin='*' --NotebookApp.token='colab_gpu_2026' --NotebookApp.disable_check_xsrf=True > /tmp/jup.log 2>&1 & sleep 3 && nohup cloudflared tunnel --url http://127.0.0.1:9999 > /tmp/cf.log 2>&1 & sleep 10 && grep trycloudflare /tmp/cf.log



^C


In [ ]:
!kill -9 $(lsof -t -i:9999) 2>/dev/null; nohup jupyter notebook --port=9999 --no-browser --ip=0.0.0.0 --NotebookApp.allow_origin='*' --NotebookApp.token='colab_gpu_2026' --NotebookApp.disable_check_xsrf=True > /tmp/jupyter.log 2>&1 &


In [ ]:
!sleep 5 && grep trycloudflare /tmp/cf.log

grep: /tmp/cf.log: No such file or directory


In [ ]:
!cat /tmp/jupyter.log

In [ ]:
!ps aux | grep cloudflared; ls -l /tmp/cf.log /tmp/jup.log

root        8350  0.0  0.0   7376  3516 ?        S    18:49   0:00 /bin/bash -c ps aux | grep cloudflared; ls -l /tmp/cf.log /tmp/jup.log
root        8352  0.0  0.0   6484  2460 ?        S    18:49   0:00 grep cloudflared
ls: cannot access '/tmp/cf.log': No such file or directory
ls: cannot access '/tmp/jup.log': No such file or directory


In [ ]:
!ps aux | grep jupyter

root         117  0.3  1.1 413576 155504 ?       Sl   18:18   0:06 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root        1320  1.7  6.1 10316736 817224 ?     Ssl  18:23   0:23 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-0760749b-88ec-4c42-bf8d-f16871f3dd1f.json
root        4930  0.6  0.0      0     0 ?        Z    18:36   0:03 [jupyter-noteboo] <defunct>
root        7379  0.0  0.1  31276 21848 ?        R    18:45   0:00 /usr/bin/python3 /usr/local/bin/jupyter-notebook --port=9999 --no-browser --ip=0.0.0.0 --NotebookApp.allow_origin=* --NotebookApp.token=colab_gpu_2026 --NotebookApp.disable_check_xsrf=True
root        7381  0.0  0.0   7376  3480 ?        S    18:45 

In [ ]:
!pkill -f cloudflared; pkill -f jupyter-notebook; sleep 2; nohup jupyter notebook --port=9999 --no-browser --ip=0.0.0.0 --NotebookApp.allow_origin='*' --NotebookApp.token='colab_gpu_2026' --NotebookApp.disable_check_xsrf=True > /tmp/jup.log 2>&1 & sleep 3 && nohup cloudflared tunnel --url http://127.0.0.1:9999 > /tmp/cf.log 2>&1 & sleep 10 && grep trycloudflare /tmp/cf.log

^C


In [ ]:
!nohup cloudflared tunnel --url http://127.0.0.1:9999 > /tmp/cloudflared.log 2>&1 &

In [ ]:
!curl -s http://localhost:9999/api/status?token=colab_gpu_2026

In [ ]:
!nohup cloudflared tunnel --url http://127.0.0.1:9999 > /tmp/cf.log 2>&1 & sleep 10 && grep trycloudflare /tmp/cf.log

2026-03-12T18:50:49Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-12T18:50:53Z INF |  https://charming-spec-pierce-organic.trycloudflare.com                                    |


In [ ]:
!curl -v http://localhost:9999/api/status?token=colab_gpu_2026 2>&1 | head -30


*   Trying 127.0.0.1:9999...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0* connect to 127.0.0.1 port 9999 failed: Connection refused
*   Trying ::1:9999...
* connect to ::1 port 9999 failed: Connection refused
* Failed to connect to localhost port 9999 after 0 ms: Connection refused
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
* Closing connection 0
curl: (7) Failed to connect to localhost port 9999 after 0 ms: Connection refused


In [ ]:
!curl -s http://localhost:8888/?token=colab_gpu_2026 2>&1 | head -5; echo "---"; netstat -tlnp 2>/dev/null | grep -E '(9999|8888)'; echo "---"; ps aux | grep jupyter

---
---
root         117  0.3  1.1 413576 155548 ?       Sl   18:18   0:06 /usr/bin/python3 /usr/local/bin/jupyter-server --debug --transport="ipc" --ip=172.28.0.12 --ServerApp.token= --port=9000 --FileContentsManager.root_dir=/ --FileContentsManager.allow_hidden=True --ServerApp.log_format="|%(levelname)s|%(message)s" --ServerApp.iopub_data_rate_limit=1e10 --MappingKernelManager.root_dir=/content
root        1320  1.3  6.1 10316736 817232 ?     Ssl  18:23   0:24 /usr/bin/python3 -m colab_kernel_launcher -f /root/.local/share/jupyter/runtime/kernel-0760749b-88ec-4c42-bf8d-f16871f3dd1f.json
root        4930  0.3  0.0      0     0 ?        Z    18:36   0:03 [jupyter-noteboo] <defunct>
root        9168  0.0  0.0   7376  3412 ?        S    18:52   0:00 /bin/bash -c curl -s http://localhost:8888/?token=colab_gpu_2026 2>&1 | head -5; echo "---"; netstat -tlnp 2>/dev/null | grep -E '(9999|8888)'; echo "---"; ps aux | grep jupyter
root        9174  0.0  0.0   6484  2572 ?        S    18:52   0

In [ ]:
!pkill -f 'port=9999'; nohup jupyter notebook --ip=0.0.0.0 --port=9999 --no-browser --NotebookApp.token='colab_gpu_2026' --NotebookApp.allow_origin='*' --NotebookApp.disable_check_xsrf=True > /tmp/jup.log 2>&1 & sleep 5; netstat -tlnp | grep 9999; echo "---"; pkill cloudflared; nohup cloudflared tunnel --url http://127.0.0.1:9999 > /tmp/cf2.log 2>&1 & sleep 15; cat /tmp/cf2.log | grep -i 'trycloudflare'

^C


In [ ]:
!echo "Jupyter Log:"; tail -n 5 /tmp/jup.log; echo "---"; echo "Cloudflared Log:"; tail -n 20 /tmp/cf2.log; echo "---"; netstat -tlnp | grep 9999; echo "---"; grep -i trycloudflare /tmp/cf2.log

Jupyter Log:
tail: cannot open '/tmp/jup.log' for reading: No such file or directory
---
Cloudflared Log:
tail: cannot open '/tmp/cf2.log' for reading: No such file or directory
---
---
grep: /tmp/cf2.log: No such file or directory


In [ ]:
!pkill -f 'port=9999'; pkill cloudflared; nohup jupyter notebook --ip=0.0.0.0 --port=9999 --no-browser --NotebookApp.token='colab_gpu_2026' --NotebookApp.allow_origin='*' --NotebookApp.disable_check_xsrf=True > /tmp/jup2.log 2>&1 & nohup cloudflared tunnel --url http://127.0.0.1:9999 > /tmp/cf3.log 2>&1 & sleep 20; grep -i trycloudflare /tmp/cf3.log

^C


In [ ]:
!echo "CF3 Log:"; cat /tmp/cf3.log; echo "---"; netstat -tlnp | grep 9999

CF3 Log:
cat: /tmp/cf3.log: No such file or directory
---


In [ ]:
import os
os.system("pkill -f 'port=9999'")
os.system("pkill cloudflared")
os.system("nohup jupyter notebook --ip=0.0.0.0 --port=9999 --no-browser --NotebookApp.token='colab_gpu_2026' --NotebookApp.allow_origin='*' --NotebookApp.disable_check_xsrf=True > /tmp/jup4.log 2>&1 &")
os.system("nohup cloudflared tunnel --url http://127.0.0.1:9999 > /tmp/cf4.log 2>&1 &")
print("Done")

Done


In [ ]:
!grep -i trycloudflare /tmp/cf4.log

2026-03-12T18:55:28Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-12T18:55:33Z INF |  https://experiment-seekers-brakes-hero.trycloudflare.com                                  |


In [ ]:
!curl -v http://localhost:9999/api/status?token=colab_gpu_2026 2>&1 | grep -E "HTTP/|{"}"

/bin/bash: -c: line 1: unexpected EOF while looking for matching `"'
/bin/bash: -c: line 2: syntax error: unexpected end of file


In [ ]:
!curl -s http://localhost:9999/api/status?token=colab_gpu_2026
echo "---"
tail -n 5 /tmp/jup4.log

SyntaxError: invalid syntax (637841228.py, line 2)

In [ ]:
!curl -s http://127.0.0.1:9999/api?token=colab_gpu_2026


In [ ]:
!ss -tlnp | grep 9999

In [ ]:
!jupyter notebook --no-browser --port=9999 --ip=0.0.0.0 --NotebookApp.token='colab_gpu_2026' --NotebookApp.allow_origin='*' --NotebookApp.disable_check_xsrf=True &


  _   _          _      _
 | | | |_ __  __| |__ _| |_ ___
 | |_| | '_ \/ _` / _` |  _/ -_)
  \___/| .__/\__,_\__,_|\__\___|
       |_|
                       
Read the migration plan to Notebook 7 to learn about the new features and the actions to take if you are using extensions.

https://jupyter-notebook.readthedocs.io/en/latest/migrate_to_notebook7.html

Please note that updating to Notebook 7 might break some of your extensions.

[I 18:59:39.947 NotebookApp] Loading IPython parallel extension
jupyter_http_over_ws extension initialized. Listening on /http_over_websocket
[I 18:59:40.035 NotebookApp] [Jupytext Server Extension] Deriving a TextFileContentsManager from LargeFileManager
[C 18:59:41.000 NotebookApp] Running as root is not recommended. Use --allow-root to bypass.


In [ ]:
!jupyter notebook --no-browser --port=9999 --ip=0.0.0.0 --allow-root --NotebookApp.token='colab_gpu_2026' --NotebookApp.allow_origin='*' --NotebookApp.disable_check_xsrf=True &


  _   _          _      _
 | | | |_ __  __| |__ _| |_ ___
 | |_| | '_ \/ _` / _` |  _/ -_)
  \___/| .__/\__,_\__,_|\__\___|
       |_|
                       
Read the migration plan to Notebook 7 to learn about the new features and the actions to take if you are using extensions.

https://jupyter-notebook.readthedocs.io/en/latest/migrate_to_notebook7.html

Please note that updating to Notebook 7 might break some of your extensions.

[I 19:03:21.299 NotebookApp] Loading IPython parallel extension
jupyter_http_over_ws extension initialized. Listening on /http_over_websocket
[I 19:03:21.429 NotebookApp] [Jupytext Server Extension] Deriving a TextFileContentsManager from LargeFileManager
[I 19:03:22.788 NotebookApp] The port 9999 is already in use, trying another port.
[I 19:03:22.789 NotebookApp] Serving notebooks from local directory: /content
[I 19:03:22.789 NotebookApp] Jupyter Notebook 6.5.7 is running at:
[I 19:03:22.789 NotebookApp] http://695ee33a482f:10000/?token=...
[I 19:03:22

KeyboardInterrupt: 

In [ ]:
!ss -tlnp | grep 9999
!curl -s http://127.0.0.1:9999/api?token=colab_gpu_2026

LISTEN 0      128          0.0.0.0:9999       0.0.0.0:*    users:(("jupyter-noteboo",pid=11220,fd=7))
{"version": "6.5.7"}

In [ ]:
!ps aux | grep cloudflared | grep -v grep

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:9999 2>&1 | grep -m1 trycloudflare

2026-03-12T19:09:04Z INF Requesting new quick Tunnel on trycloudflare.com...


In [ ]:
!nohup cloudflared tunnel --url http://127.0.0.1:9999 > /tmp/cf5.log 2>&1 & sleep 15; cat /tmp/cf5.log

2026-03-12T19:09:24Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-03-12T19:09:24Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-03-12T19:09:28Z INF +--------------------------------------------------------------------------------------------+
2026-03-12T19:09:28Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-03-12T19:09:28Z INF |  https://mile-rod-viewers-proper.trycloudflare.com    